## Transferir o notebook

In [1]:
%pip install -r requirements.txt

import kagglehub
path = kagglehub.dataset_download( "harlfoxem/housesalesprediction",                                                                   output_dir="data/kingcounty",)

print("Ficheiros do dataset em:", path)


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: /Users/anytng/Developer/Universidade/Masters/FundamentalsOfAI/FAI26_P02_G09/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/anytng/Developer/Universidade/Masters/FundamentalsOfAI/FAI26_P02_G09/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ficheiros do dataset em: data/kingcounty


## Mudança de dataset

Originalmente usámos o Melbourne Housing Snapshot, mas trocámos para o King County House Sales porque em Melbourne metade das vendas não tinha área construída e não havia informação sobre o estado e a qualidade das casas. O King County não tem valores em falta e inclui o estado, a qualidade da construção e as renovações.

## EDA

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

df = pd.read_csv("data/kingcounty/kc_house_data.csv")
df["date"] = pd.to_datetime(df["date"].str[:8])  # as datas vêm no formato 20141013T000000

print(f"{df.shape[0]} linhas x {df.shape[1]} colunas")
print("Vendas entre", df["date"].min().date(), "e", df["date"].max().date())
df.head()

21613 linhas x 21 colunas
Vendas entre 2014-05-02 e 2015-05-27


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,2014-10-13,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,2014-12-09,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,2015-02-25,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,2014-12-09,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,2015-02-18,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


Os dados são 21 613 vendas de casas no condado de King, no estado de Washington, que inclui Seattle, entre maio de 2014 e maio de 2015. Vêm do dataset [House Sales in King County, USA](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction) no Kaggle, feito a partir dos registos públicos de vendas do condado.

As áreas estão em pés quadrados, e 1 sqft são cerca de 0,093 m². `sqft_living` é a área da casa, `sqft_lot` a do terreno, e `sqft_above` e `sqft_basement` são a área acima do chão e a da cave. `sqft_living15` e `sqft_lot15` são as mesmas áreas nas 15 casas vizinhas mais próximas. `grade` é a qualidade da construção numa escala de 1 a 13 e `condition` é o estado da casa de 1 a 5, as duas atribuídas pelo condado. `view` vai de 0 a 4 e diz quão boa é a vista, `waterfront` é 1 quando a casa fica à beira de água e `yr_renovated` é 0 quando a casa nunca foi renovada.

In [3]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nao_nulos": df.notna().sum(),
    "distintos": df.nunique(),
    "exemplo": df.iloc[0],
})
overview

,dtype,nao_nulos,distintos,exemplo
id,int64,21613,21436,7129300520
date,datetime64[us],21613,372,2014-10-13 00:00:00
price,float64,21613,4028,221900.0
bedrooms,int64,21613,13,3
bathrooms,float64,21613,30,1.0
sqft_living,int64,21613,1038,1180
sqft_lot,int64,21613,9782,5650
floors,float64,21613,6,1.0
waterfront,int64,21613,2,0
view,int64,21613,5,0


In [4]:
df.describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2)

,count,mean,min,1%,25%,50%,75%,99%,max,std
id,21613.0,4580301520.864988,1000102.0,122033870.92,2123049194.0,3904930410.0,7308900445.0,9558728047.200001,9900000190.0,2876565571.312057
date,21613,2014-10-29 04:38:01.959931,2014-05-02 00:00:00,2014-05-06 00:00:00,2014-07-22 00:00:00,2014-10-16 00:00:00,2015-02-17 00:00:00,2015-05-07 00:00:00,2015-05-27 00:00:00,NaN
price,21613.0,540088.141767,75000.0,153500.36,321950.0,450000.0,645000.0,1964400.0,7700000.0,367127.196483
bedrooms,21613.0,3.370842,0.0,2.0,3.0,3.0,4.0,6.0,33.0,0.930062
bathrooms,21613.0,2.114757,0.0,1.0,1.75,2.25,2.5,4.25,8.0,0.770163
sqft_living,21613.0,2079.899736,290.0,720.0,1427.0,1910.0,2550.0,4978.8,13540.0,918.440897
sqft_lot,21613.0,15106.967566,520.0,1013.12,5040.0,7618.0,10688.0,213008.0,1651359.0,41420.511515
floors,21613.0,1.494309,1.0,1.0,1.0,1.5,2.0,3.0,3.5,0.539989
waterfront,21613.0,0.007542,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.086517
view,21613.0,0.234303,0.0,0.0,0.0,0.0,0.0,4.0,4.0,0.766318


In [5]:
# Preço mediano pela qualidade da construção, pelo estado, pela vista e por estar à beira de água
for col in ["grade", "condition", "view", "waterfront"]:
    display(df.groupby(col)["price"].agg(["count", "median"]).round(0).T)

grade,1,3,4,5,6,7,8,9,10,11,12,13
count,1.0,3.0,29.0,242.0,2038.0,8981.0,6068.0,2615.0,1134.0,399.0,90.0,13.0
median,142000.0,262000.0,205000.0,228700.0,275276.0,375000.0,510000.0,720000.0,914327.0,1284000.0,1817500.0,2983000.0


condition,1,2,3,4,5
count,30.0,172.0,14031.0,5679.0,1701.0
median,262500.0,279000.0,450000.0,440000.0,526000.0


view,0,1,2,3,4
count,19489.0,332.0,963.0,510.0,319.0
median,432500.0,690944.0,675000.0,802500.0,1185000.0


waterfront,0,1
count,21450.0,163.0
median,450000.0,1400000.0


In [6]:
# Preço por trimestre
df.groupby(df["date"].dt.to_period("Q"))["price"].agg(["count", "mean", "median"]).round(0)

,count,mean,median
date,,,
2014Q2,3948,553559.0,465000.0
2014Q3,5925,537406.0,450000.0
2014Q4,4760,529467.0,438975.0
2015Q1,4103,528655.0,440000.0
2015Q2,2877,561005.0,470500.0


In [7]:
# Os 5 códigos postais mais baratos e os 5 mais caros, pela mediana do preço
por_zip = df.groupby("zipcode")["price"].agg(["count", "median"]).sort_values("median")
print(f"{len(por_zip)} códigos postais")
pd.concat([por_zip.head(), por_zip.tail()])

70 códigos postais


,count,median
zipcode,,
98002,199,235000.0
98168,269,235000.0
98032,125,249000.0
98001,362,260000.0
98188,136,264000.0
98005,168,765475.0
98112,269,915000.0
98040,282,993750.0
98004,317,1150000.0


Não há valores em falta nem linhas repetidas. A área da casa é sempre a soma da área acima do chão com a da cave, por isso `sqft_living`, `sqft_above` e `sqft_basement` repetem informação.

In [8]:
print("Valores em falta:", df.isna().sum().sum())
print("Linhas exatamente iguais:", df.duplicated().sum())
print("Mesmo imóvel na mesma data:", df.duplicated(["id", "date"]).sum())
print("sqft_living = sqft_above + sqft_basement em todas as linhas:",
      (df["sqft_living"] == df["sqft_above"] + df["sqft_basement"]).all())

Valores em falta: 0
Linhas exatamente iguais: 0
Mesmo imóvel na mesma data: 0
sqft_living = sqft_above + sqft_basement em todas as linhas: True


Uma casa é identificada pelo `id`. 176 casas foram vendidas duas vezes no período, uma delas três vezes, e as características nunca mudam entre vendas. Entre a primeira e a última venda passaram em mediana 178 dias e o preço subiu em mediana 54%. Uma subida destas em seis meses, sem mudar `grade`, `condition` nem `yr_renovated`, sugere casas compradas para obras e revendidas, com as obras por registar.

In [9]:
repetidos = df[df.duplicated("id", keep=False)].sort_values(["id", "date"])
print("Imóveis vendidos mais do que uma vez:", repetidos["id"].nunique(), f"({len(repetidos)} linhas)")

# Algum imóvel muda de características entre vendas?
fisicas = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "condition", "grade", "yr_renovated"]
print("Com características diferentes entre vendas:", (repetidos.groupby("id")[fisicas].nunique() > 1).any(axis=1).sum())

evolucao = repetidos.groupby("id").agg(primeira_data=("date", "first"), ultima_data=("date", "last"),
                                       primeiro_preco=("price", "first"), ultimo_preco=("price", "last"))
evolucao["dias"] = (evolucao["ultima_data"] - evolucao["primeira_data"]).dt.days
evolucao["variacao_pct"] = ((evolucao["ultimo_preco"] / evolucao["primeiro_preco"] - 1) * 100).round(1)
evolucao[["dias", "variacao_pct"]].describe().round(1)

Imóveis vendidos mais do que uma vez: 176 (353 linhas)
Com características diferentes entre vendas: 0


,dias,variacao_pct
count,176.0,176.0
mean,183.3,56.8
std,67.0,46.7
min,61.0,-5.4
25%,128.5,23.6
50%,177.5,54.4
75%,233.8,74.2
max,315.0,321.8


Procuramos valores impossíveis ou pouco prováveis numa casa.

In [10]:
checks = {
    "bedrooms == 0": df["bedrooms"] == 0,
    "bathrooms == 0": df["bathrooms"] == 0,
    "bedrooms >= 10": df["bedrooms"] >= 10,
    "sqft_living < 500": df["sqft_living"] < 500,
    "sqft_lot > 435600 (10 acres)": df["sqft_lot"] > 435_600,
    "grade <= 4": df["grade"] <= 4,
    "yr_built > ano da venda": df["yr_built"] > df["date"].dt.year,
    "price < 100k": df["price"] < 100_000,
}
pd.Series({k: v.sum() for k, v in checks.items()}, name="linhas").to_frame()

,linhas
bedrooms == 0,13
bathrooms == 0,10
bedrooms >= 10,5
sqft_living < 500,17
sqft_lot > 435600 (10 acres),27
grade <= 4,33
yr_built > ano da venda,12
price < 100k,25


In [11]:
colunas = ["price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "grade", "yr_built"]
df.loc[checks["bedrooms >= 10"] | checks["bedrooms == 0"] | checks["bathrooms == 0"], colunas].sort_values("bedrooms")

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,grade,yr_built
875,1095000.0,0,0.00,3064,4764,3.5,7,1990
18379,265000.0,0,0.75,384,213444,1.0,4,2003
14423,139950.0,0,0.00,844,4269,1.0,7,1913
12653,320000.0,0,2.50,1490,7111,2.0,7,1999
9854,235000.0,0,0.00,1470,4800,2.0,7,1996
8484,240000.0,0,2.50,1810,5669,2.0,7,2003
8477,339950.0,0,2.50,2290,8319,2.0,8,1985
9773,355000.0,0,0.00,2460,8049,2.0,8,1990
4868,228000.0,0,1.00,390,5900,1.0,4,1953
3467,288000.0,0,1.50,1430,1650,3.0,7,1999


In [12]:
df.loc[checks["sqft_living < 500"] | checks["grade <= 4"] | checks["price < 100k"], colunas + ["condition"]].sort_values("sqft_living")

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,grade,yr_built,condition
19452,142000.0,0,0.00,290,20875,1.0,1,1963,1
15381,276000.0,1,0.75,370,1801,1.0,5,1923,5
860,245000.0,1,0.75,380,15000,1.0,5,1963,3
18379,265000.0,0,0.75,384,213444,1.0,4,2003,3
4868,228000.0,0,1.00,390,5900,1.0,4,1953,2
...,...,...,...,...,...,...,...,...,...
5303,99000.0,2,1.00,960,8236,1.0,6,1948,2
18939,90000.0,3,1.00,980,2490,2.0,6,1969,4
7973,120000.0,2,1.00,990,39964,1.0,4,1945,2
10585,95000.0,2,1.00,1070,20450,1.0,6,1948,2


O único erro claro é a casa com 33 quartos em 1620 sqft e 1,75 casas de banho, que deve ter 3. As casas com 0 quartos ou 0 casas de banho são poucas e têm áreas normais, por isso devem ser valores por registar. As casas muito pequenas, com `grade` baixo ou vendidas abaixo de 100 mil dólares são sobretudo casas antigas e de construção simples, de 1945 em mediana, o que é plausível. Os terrenos acima de 10 acres, cerca de 4 hectares, ficam em zonas rurais do condado como Enumclaw, Vashon e Carnation. As 12 casas construídas depois da data da venda foram vendidas antes de estarem acabadas. Fora os 33 quartos, mantemos estes valores.

## Preparação dos dados

Corrigimos só o que é um erro claro, os 33 quartos, que passam a 3.

In [13]:
dados = df.copy()

# 33 quartos numa casa de 1620 sqft com 1,75 casas de banho é um erro de escrita de 3
erro_quartos = dados["bedrooms"] == 33
dados.loc[erro_quartos, "bedrooms"] = 3
print(f"bedrooms corrigidos: {erro_quartos.sum()} | linhas finais: {len(dados)}")

bedrooms corrigidos: 1 | linhas finais: 21613


Para saber se uma casa foi vendida acima ou abaixo do normal, estimamos o preço esperado com uma regressão e comparamos com o preço real. O alvo é o log do preço, porque o preço é muito assimétrico e em log o erro passa a ser relativo.

A regressão usa o tamanho e a localização da casa, ou seja `sqft_living`, `sqft_lot`, `bedrooms`, `bathrooms`, o código postal e o mês da venda. As áreas entram como curvas, porque o preço não cresce em linha reta com a área. Os quartos, as casas de banho, o código postal e o mês entram como categorias, com uma coluna de 0 e 1 para cada valor. O código postal como número não quer dizer nada, mas como categoria cada zona fica com o seu próprio nível de preço. Não usamos a qualidade, o estado, a vista, o ano de construção nem as renovações, porque são as variáveis com que queremos explicar o desvio ao preço esperado.

Usamos uma regressão Ridge, uma regressão linear com uma pequena penalização, porque há mais de 100 colunas criadas a partir do código postal, do mês e das curvas. O preço esperado de cada venda vem de validação cruzada com 5 partes, de um modelo que não viu essa venda. Assim um preço anormal não puxa o modelo para si.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer

# Áreas em log, porque o preço não cresce em linha reta com a área
dados["LogLiving"] = np.log(dados["sqft_living"])
dados["LogLot"] = np.log(dados["sqft_lot"])

# Quartos e casas de banho como categorias, com 6 ou mais quartos e 5 ou mais casas de banho juntos
dados["QuartosCat"] = dados["bedrooms"].clip(upper=6).astype(str)
dados["BanhosCat"] = dados["bathrooms"].clip(upper=5).round().astype(int).astype(str)
dados["CodigoPostal"] = dados["zipcode"].astype(str)
dados["MesVenda"] = dados["date"].dt.to_period("M").astype(str)

spline_cols = ["LogLiving", "LogLot"]
cat_cols = ["QuartosCat", "BanhosCat", "CodigoPostal", "MesVenda"]
colunas_auxiliares = spline_cols + cat_cols

modelo = make_pipeline(
    ColumnTransformer([
        ("spline", SplineTransformer(n_knots=6, degree=3), spline_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]),
    Ridge(alpha=1.0),
)

X, y = dados[spline_cols + cat_cols], np.log(dados["price"])
y_prev = cross_val_predict(modelo, X, y, cv=KFold(5, shuffle=True, random_state=42))

r2 = 1 - ((y - y_prev) ** 2).sum() / ((y - y.mean()) ** 2).sum()
erro_pct = (np.exp(np.abs(y - y_prev)) - 1) * 100
print(f"R² (log do preço, out-of-fold): {r2:.3f}")
print(f"Erro mediano: {np.median(erro_pct):.1f}% do preço")

R² (log do preço, out-of-fold): 0.841
Erro mediano: 11.8% do preço


O desvio de cada venda é a diferença, em %, entre o preço real e o esperado. O limite é 12%, perto do erro mediano da regressão, para que uma venda só seja `alto` ou `baixo` quando se afasta do esperado mais do que a regressão costuma errar. Abaixo de −12% a venda é `baixo`, acima de +12% é `alto` e entre os dois é `normal`.

In [15]:
dados["PrecoEsperado"] = np.exp(y_prev).round(0)
dados["DesvioPreco"] = (dados["price"] / dados["PrecoEsperado"] - 1) * 100  # em %

LIMITE = 12  # %
dados["ClassePreco"] = pd.cut(dados["DesvioPreco"], [-np.inf, -LIMITE, LIMITE, np.inf],
                              labels=["baixo", "normal", "alto"])

print(f"baixo: desvio < -{LIMITE}% | normal: entre -{LIMITE}% e +{LIMITE}% | alto: desvio > +{LIMITE}%")
contagem = dados["ClassePreco"].value_counts().to_frame()
contagem["pct_vendas"] = (dados["ClassePreco"].value_counts(normalize=True) * 100).round(1)
contagem

baixo: desvio < -12% | normal: entre -12% e +12% | alto: desvio > +12%


,count,pct_vendas
ClassePreco,,
normal,11431,52.9
alto,5335,24.7
baixo,4847,22.4


In [16]:
# Desvio por classe
dados.groupby("ClassePreco", observed=True)["DesvioPreco"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
ClassePreco,,,,,,,,
baixo,4847.0,-23.0,9.6,-72.6,-27.7,-20.6,-15.6,-12.0
normal,11431.0,0.1,6.6,-12.0,-5.4,0.1,5.6,12.0
alto,5335.0,29.9,23.9,12.0,16.3,22.3,33.7,251.3


In [17]:
# Percentagem de cada classe pela qualidade da construção, pelo estado e pela vista
for col in ["grade", "condition", "view"]:
    display((pd.crosstab(dados[col], dados["ClassePreco"], normalize="index") * 100).round(1))

ClassePreco,baixo,normal,alto
grade,,,
1,100.0,0.0,0.0
3,33.3,0.0,66.7
4,51.7,10.3,37.9
5,45.0,31.8,23.1
6,35.5,42.3,22.2
7,25.5,55.7,18.8
8,18.5,56.2,25.3
9,11.9,52.9,35.2
10,17.5,44.5,38.0


ClassePreco,baixo,normal,alto
condition,,,
1,66.7,13.3,20.0
2,52.3,27.9,19.8
3,22.5,54.1,23.4
4,23.2,51.8,25.0
5,15.6,49.6,34.7


ClassePreco,baixo,normal,alto
view,,,
0,23.4,54.9,21.6
1,18.4,38.9,42.8
2,15.5,39.4,45.2
3,11.8,32.9,55.3
4,4.1,14.4,81.5


Cerca de metade das vendas são `normal` e o resto divide-se quase por igual entre `baixo` e `alto`. A qualidade da construção, o estado e a vista mudam bastante a proporção das classes. 82% das casas com a melhor vista e 59% das casas com `grade` 12 são `alto`, e mais de metade das casas em estado 1 ou 2 são `baixo`.

In [18]:
# Exemplos: as vendas mais acima e mais abaixo do esperado
cols = ["zipcode", "bedrooms", "bathrooms", "sqft_living", "grade", "condition", "price", "PrecoEsperado", "DesvioPreco", "ClassePreco"]
display(dados.nlargest(5, "DesvioPreco")[cols].round(1))
dados.nsmallest(5, "DesvioPreco")[cols].round(1)

,zipcode,bedrooms,bathrooms,sqft_living,grade,condition,price,PrecoEsperado,DesvioPreco,ClassePreco
1081,98178,2,1.0,1150,6,4,825000.0,234818.0,251.3,alto
20325,98056,3,3.5,4410,11,3,3000000.0,872660.0,243.8,alto
6402,98118,5,2.5,3750,10,3,2903000.0,859557.0,237.7,alto
16252,98034,3,2.5,2510,10,3,1950000.0,591002.0,229.9,alto
4013,98075,3,2.2,3000,11,3,2400000.0,737682.0,225.3,alto


,zipcode,bedrooms,bathrooms,sqft_living,grade,condition,price,PrecoEsperado,DesvioPreco,ClassePreco
16893,98117,3,1.0,1230,6,1,125000.0,456616.0,-72.6,baixo
18332,98116,3,1.0,1200,7,1,130000.0,447417.0,-70.9,baixo
1222,98033,3,1.0,1110,7,4,130000.0,446067.0,-70.9,baixo
21372,98119,5,3.5,4460,10,3,490000.0,1611082.0,-69.6,baixo
14835,98119,3,2.5,2370,9,3,300523.0,945808.0,-68.2,baixo


## Guardar o dataset preparado

A pasta `data/` está no `.gitignore`, por isso o ficheiro é gerado ao correr este notebook.

In [19]:
saida = "data/kingcounty/kc_clean.csv"
dados.drop(columns=colunas_auxiliares).to_csv(saida, index=False)
print("Guardado em", saida, dados.shape)

Guardado em data/kingcounty/kc_clean.csv (21613, 30)
